In [2]:
from __future__ import annotations

"""
09_sota_comparison_full.py

Full SOTA training/evaluation script for FruitNet after RL refinement.
This replaces the lightweight aggregator-style 09_sota_comparison.py.

What this script does:
1) Resolves fine-tuned checkpoints for baseline/FruitNet/SOTA models.
2) Trains missing checkpoints when RUN_TRAINING=1.
3) Supports resume from last.pt when a training run was interrupted.
4) Validates each model on the requested split and logs detection metrics.
5) Computes model params/GFLOPs from each final checkpoint.
6) Runs fixed-threshold counting evaluation for each detector.
7) Adds FruitNet GCSR row from RL output generated by 07_rl_count_refine.py.
8) Writes full status/progress/checkpoint manifests to survive crashes.

Notes:
- YOLOv10n / YOLO11n are included as SOTA baselines when available in your
  Ultralytics install/cache. The script tries multiple aliases and logs skips.
- Exact Pepper-YOLO / GAE-YOLO are only run if you provide PEPPER_YOLO_PATH and
  GAE_YOLO_PATH. Otherwise they are written to audit logs as skipped, because
  reporting them without the real implementation would be scientifically unsafe.
"""

from pathlib import Path
from datetime import datetime
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
import os
import sys
import json
import time
import math
import shutil
import logging
import traceback
import platform
import hashlib
import subprocess

import numpy as np
import pandas as pd

try:
    import yaml
except Exception as e:  # pragma: no cover
    raise RuntimeError("Please install pyyaml: pip install pyyaml") from e

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# ============================================================
# 0. Generic helpers
# ============================================================

def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if raw == "":
        return default
    return raw in {"1", "true", "yes", "y", "on"}


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else default


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else default


def parse_csv_list(raw: str, default: Sequence[str]) -> List[str]:
    raw = raw.strip()
    if not raw:
        return list(default)
    return [x.strip() for x in raw.split(",") if x.strip()]


def now() -> str:
    return datetime.now().isoformat(timespec="seconds")


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if c.name == "fruitnet-chili-yield":
            return c
    return here


def atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding=encoding)
    os.replace(tmp, path)


def atomic_write_json(path: Path, obj: Any, indent: int = 2) -> None:
    atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=indent))


def atomic_write_csv(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def run_cmd(cmd: Sequence[str], logger: Optional[logging.Logger] = None, timeout: Optional[int] = None) -> Tuple[int, str]:
    try:
        p = subprocess.run(list(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout)
        out = p.stdout or ""
        if logger:
            logger.info("CMD %s -> rc=%s\n%s", " ".join(cmd), p.returncode, out[-3000:])
        return p.returncode, out
    except Exception as e:
        if logger:
            logger.warning("CMD failed %s: %s", cmd, repr(e))
        return -1, repr(e)


# ============================================================
# 1. Paths and config
# ============================================================

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path(os.environ.get("DATA_DIR", "/mnt/f/fruitnet-chili-yield-data/finetune_data")).resolve()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", "/mnt/f/fruitnet-chili-yield-outputs")).resolve()
DATASET_NAME = DATA_DIR.name

EXP_NAME = os.environ.get("EXP_NAME", f"exp05_sota_comparison_{DATASET_NAME}")
EXP_DIR = OUTPUT_ROOT / "results" / EXP_NAME
LOG_DIR = OUTPUT_ROOT / "logs"
LOCAL_LOG_DIR = PROJECT_ROOT / "logs"
RUN_DIR = OUTPUT_ROOT / "runs" / "detectors" / DATASET_NAME
VAL_DIR = OUTPUT_ROOT / "runs" / "detectors" / f"{DATASET_NAME}_sota_val"
ARTIFACT_DIR = OUTPUT_ROOT / "artifacts" / "finetuned"
SOTA_ARTIFACT_DIR = OUTPUT_ROOT / "artifacts" / "sota_finetuned"
PROGRESS_DIR = EXP_DIR / "progress"
COUNT_DIR = EXP_DIR / "per_model_count_eval"
REGISTRY_DIR = EXP_DIR / "registry"

for d in [EXP_DIR, LOG_DIR, LOCAL_LOG_DIR, RUN_DIR, VAL_DIR, ARTIFACT_DIR, SOTA_ARTIFACT_DIR, PROGRESS_DIR, COUNT_DIR, REGISTRY_DIR, PROJECT_ROOT / "configs" / "data"]:
    d.mkdir(parents=True, exist_ok=True)

SCRIPT_TAG = "09_sota_comparison_full"
LOG_PATH = LOG_DIR / f"{SCRIPT_TAG}_{DATASET_NAME}_{datetime.now():%Y%m%d_%H%M%S}.log"
LOCAL_LOG_PATH = LOCAL_LOG_DIR / f"{SCRIPT_TAG}_{DATASET_NAME}_{datetime.now():%Y%m%d_%H%M%S}.log"
LOGGER = logging.getLogger("fruitnet_09_full")
LOGGER.setLevel(logging.INFO)
LOGGER.handlers.clear()
_fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
for _p in [LOG_PATH, LOCAL_LOG_PATH]:
    _fh = logging.FileHandler(_p, encoding="utf-8")
    _fh.setFormatter(_fmt)
    LOGGER.addHandler(_fh)
_sh = logging.StreamHandler(sys.stdout)
_sh.setFormatter(_fmt)
LOGGER.addHandler(_sh)

CLASS_MODE = os.environ.get("CLASS_MODE", "single_chili").strip()
IMG_SIZE = env_int("IMG_SIZE", 640)
BATCH = env_int("BATCH", 8)
EPOCHS = env_int("SOTA_EPOCHS", env_int("EPOCHS", 100))
WORKERS = env_int("WORKERS", 8)
DEVICE = os.environ.get("DEVICE", "0")
PATIENCE = env_int("PATIENCE", 30)
SEED = env_int("SEED", 42)
RUN_TRAINING = env_bool("RUN_TRAINING", True)
RUN_VALIDATION = env_bool("RUN_VALIDATION", True)
RUN_COUNT_EVAL = env_bool("RUN_COUNT_EVAL", True)
FORCE_RETRAIN = env_bool("FORCE_RETRAIN", False)
RESUME_TRAINING = env_bool("RESUME_TRAINING", True)
EVAL_SPLIT_REQUESTED = os.environ.get("EVAL_SPLIT", "test").strip()
COUNT_CONF = env_float("COUNT_CONF", 0.25)
COUNT_IOU = env_float("COUNT_IOU", 0.70)
MAX_COUNT_IMAGES = env_int("MAX_COUNT_IMAGES", 0)  # 0 = all images
SOTA_ONLY = env_bool("SOTA_ONLY", False)
INCLUDE_EXTERNAL_SOTA = env_bool("INCLUDE_EXTERNAL_SOTA", True)
STRICT_EXTERNAL_SOTA = env_bool("STRICT_EXTERNAL_SOTA", False)

DEFAULT_MODEL_KEYS = [
    "yolov5s", "yolov8n", "yolov10n", "yolo11n",
    "fruitnet_b", "fruitnet_g", "fruitnet_gc", "fruitnet_gcs",
    "pepper_yolo", "gae_yolo",
]
MODEL_KEYS = parse_csv_list(os.environ.get("MODEL_KEYS", ""), DEFAULT_MODEL_KEYS)
if SOTA_ONLY:
    MODEL_KEYS = [k for k in MODEL_KEYS if k in {"yolov10n", "yolo11n", "pepper_yolo", "gae_yolo"}]

# ============================================================
# 2. Data YAML and project compatibility
# ============================================================

def default_names() -> List[str]:
    return ["chili"] if CLASS_MODE == "single_chili" else ["Chili_Green", "Chili_Transition", "Chili_Red"]


def resolve_data_yaml() -> Path:
    names: Any = default_names()
    src_yaml = DATA_DIR / "data.yaml"
    if src_yaml.exists():
        try:
            d = yaml.safe_load(src_yaml.read_text(encoding="utf-8")) or {}
            nm = d.get("names")
            if isinstance(nm, dict):
                names = [nm[k] for k in sorted(nm, key=lambda x: int(x))]
            elif isinstance(nm, list) and nm:
                names = nm
        except Exception as e:
            LOGGER.warning("Could not read %s, using default class names: %s", src_yaml, repr(e))
    out = PROJECT_ROOT / "configs" / "data" / f"{DATASET_NAME}.yaml"
    payload = {
        "path": str(DATA_DIR),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(names),
        "names": {i: n for i, n in enumerate(names)},
    }
    atomic_write_text(out, yaml.safe_dump(payload, sort_keys=False, allow_unicode=True))
    return out


DATA_YAML = resolve_data_yaml()
DATA_CFG = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8")) or {}
CLASS_NAMES = DATA_CFG.get("names", {})
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES_LIST = [CLASS_NAMES[k] for k in sorted(CLASS_NAMES, key=lambda x: int(x))]
else:
    CLASS_NAMES_LIST = list(CLASS_NAMES)
TARGET_CLS = env_int("TARGET_CLS", 0 if CLASS_MODE == "single_chili" else min(2, max(0, len(CLASS_NAMES_LIST) - 1)))


def iter_images(path: Path) -> Iterable[Path]:
    if not path.exists():
        return
    for p in sorted(path.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield p


def split_image_count(split: str) -> int:
    key = DATA_CFG.get(split)
    if not key:
        return 0
    return len(list(iter_images(Path(DATA_CFG["path"]) / str(key))))


def resolve_eval_split(preferred: str = "test") -> Optional[str]:
    seen = []
    for s in [preferred, "test", "val", "train"]:
        if s not in seen:
            seen.append(s)
            if s in {"train", "val", "test"} and split_image_count(s) > 0:
                return s
    return None


def has_train_and_val() -> bool:
    return split_image_count("train") > 0 and split_image_count("val") > 0


def register_custom_modules() -> bool:
    try:
        from models.fruitnet.register_ultralytics_modules import register_fruitnet_modules
        register_fruitnet_modules()
        LOGGER.info("Registered FruitNet custom modules.")
        return True
    except Exception as e:
        LOGGER.warning("Could not register FruitNet custom modules; FruitNet YAMLs may fail: %s", repr(e))
        return False


def yaml_needs_custom_modules(path_or_name: str) -> bool:
    p = Path(path_or_name)
    if p.exists() and p.suffix.lower() in {".yaml", ".yml"}:
        txt = p.read_text(encoding="utf-8", errors="ignore")
        return any(x in txt for x in ["CoordAtt", "CoordinateAttention", "SIoU", "GhostConv"])
    return False


# ============================================================
# 3. Model registry
# ============================================================

def path_from_env(name: str) -> Optional[Path]:
    raw = os.environ.get(name, "").strip()
    return Path(raw).expanduser().resolve() if raw else None


def existing_path_or_none(p: Optional[Path]) -> Optional[Path]:
    return p if p and p.exists() else None


def candidate_paths(model_key: str) -> List[Path]:
    names = [
        ARTIFACT_DIR / f"{model_key}_{DATASET_NAME}_best.pt",
        SOTA_ARTIFACT_DIR / f"{model_key}_{DATASET_NAME}_best.pt",
        RUN_DIR / f"{model_key}_{DATASET_NAME}" / "weights" / "best.pt",
        OUTPUT_ROOT / "runs" / "detectors" / "coco_pretrain" / model_key / "weights" / "best.pt",
    ]
    return names


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    xs = [p for p in paths if p.exists()]
    if not xs:
        return None
    # prefer newest artifact/checkpoint
    return max(xs, key=lambda x: x.stat().st_mtime)


def model_registry() -> List[Dict[str, Any]]:
    # External exact SOTA paths are only used when supplied by the user.
    pepper_path = path_from_env("PEPPER_YOLO_PATH")
    gae_path = path_from_env("GAE_YOLO_PATH")
    return [
        {
            "model_name": "YOLOv5s",
            "model_key": "yolov5s",
            "group": "baseline",
            "aliases": ["yolov5su.pt", "yolov5s.pt"],
            "trainable": True,
        },
        {
            "model_name": "YOLOv8n",
            "model_key": "yolov8n",
            "group": "baseline",
            "aliases": ["yolov8n.pt"],
            "trainable": True,
        },
        {
            "model_name": "YOLOv10n",
            "model_key": "yolov10n",
            "group": "sota",
            "aliases": ["yolov10n.pt", "yolo10n.pt"],
            "trainable": True,
        },
        {
            "model_name": "YOLO11n",
            "model_key": "yolo11n",
            "group": "sota",
            "aliases": ["yolo11n.pt", "yolov11n.pt"],
            "trainable": True,
        },
        {
            "model_name": "FruitNet B",
            "model_key": "fruitnet_b",
            "group": "fruitnet",
            "aliases": ["configs/models/fruitnet_b.yaml"],
            "trainable": True,
        },
        {
            "model_name": "FruitNet G",
            "model_key": "fruitnet_g",
            "group": "fruitnet",
            "aliases": ["configs/models/fruitnet_g.yaml"],
            "trainable": True,
        },
        {
            "model_name": "FruitNet GC",
            "model_key": "fruitnet_gc",
            "group": "fruitnet",
            "aliases": ["configs/models/fruitnet_gc.yaml"],
            "trainable": True,
        },
        {
            "model_name": "FruitNet GCS",
            "model_key": "fruitnet_gcs",
            "group": "fruitnet",
            "aliases": ["configs/models/fruitnet_gcs.yaml"],
            "trainable": True,
        },
        {
            "model_name": "Pepper-YOLO exact",
            "model_key": "pepper_yolo",
            "group": "external_sota",
            "aliases": [str(pepper_path)] if pepper_path else [],
            "external_path_required": True,
            "trainable": bool(pepper_path),
        },
        {
            "model_name": "GAE-YOLO exact",
            "model_key": "gae_yolo",
            "group": "external_sota",
            "aliases": [str(gae_path)] if gae_path else [],
            "external_path_required": True,
            "trainable": bool(gae_path),
        },
    ]


def normalize_model_key(x: Any) -> str:
    # Longest first: avoids bug mapping fruitnet_gcs/fruitnet_gc to fruitnet_g.
    s = str(x).strip().lower().replace(" ", "_").replace("-", "_")
    ordered = ["fruitnet_gcsr", "fruitnet_gcs", "fruitnet_gc", "fruitnet_g", "fruitnet_b", "yolov10n", "yolo10n", "yolo11n", "yolov11n", "yolov8n", "yolov5s", "pepper_yolo", "gae_yolo"]
    for k in ordered:
        if k in s:
            if k == "yolo10n":
                return "yolov10n"
            if k == "yolov11n":
                return "yolo11n"
            return k
    return s


def resolve_initial_source(spec: Dict[str, Any]) -> Tuple[Optional[str], str]:
    # Prefer existing fine-tuned checkpoint. If none exists, choose COCO-pretrain or model alias.
    existing = first_existing(candidate_paths(spec["model_key"]))
    if existing and not FORCE_RETRAIN:
        return str(existing), "existing_checkpoint"
    for alias in spec.get("aliases", []):
        if not alias:
            continue
        p = Path(alias)
        if p.exists():
            return str(p), "local_alias"
    # If Ultralytics supports downloading/cached weights, string alias is valid.
    for alias in spec.get("aliases", []):
        if alias.endswith(".pt"):
            return alias, "ultralytics_alias"
    if spec.get("external_path_required"):
        return None, "missing_external_path"
    return None, "missing_source"


# ============================================================
# 4. Status/progress
# ============================================================

def write_status(status: str, extra: Optional[Dict[str, Any]] = None) -> None:
    payload = {
        "status": status,
        "updated_at": now(),
        "script": Path(sys.argv[0]).name,
        "project_root": str(PROJECT_ROOT),
        "data_dir": str(DATA_DIR),
        "output_root": str(OUTPUT_ROOT),
        "dataset_name": DATASET_NAME,
        "data_yaml": str(DATA_YAML),
        "python": sys.version,
        "platform": platform.platform(),
        "log_path": str(LOG_PATH),
        "local_log_path": str(LOCAL_LOG_PATH),
    }
    if extra:
        payload.update(extra)
    atomic_write_json(EXP_DIR / "run_status.json", payload)


def load_progress() -> Dict[str, Any]:
    p = PROGRESS_DIR / "sota_progress.json"
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}


def save_progress(progress: Dict[str, Any]) -> None:
    progress["updated_at"] = now()
    atomic_write_json(PROGRESS_DIR / "sota_progress.json", progress)


def write_model_status(model_key: str, payload: Dict[str, Any]) -> None:
    payload = dict(payload)
    payload["updated_at"] = now()
    atomic_write_json(PROGRESS_DIR / f"{model_key}_status.json", payload)


# ============================================================
# 5. Metrics helpers
# ============================================================

def metrics_to_dict(m: Any) -> Dict[str, float]:
    box = getattr(m, "box", None)
    if box is None:
        return {"Precision": math.nan, "Recall": math.nan, "mAP@0.5": math.nan, "mAP@0.5:0.95": math.nan}
    def g(attr: str) -> float:
        try:
            return float(getattr(box, attr))
        except Exception:
            return math.nan
    return {"Precision": g("mp"), "Recall": g("mr"), "mAP@0.5": g("map50"), "mAP@0.5:0.95": g("map")}


def model_complexity_from_yolo(yolo_model: Any) -> Tuple[float, float]:
    try:
        info = yolo_model.info(verbose=False)
        if isinstance(info, (tuple, list)):
            # Ultralytics usually returns (layers, params, gradients, GFLOPs)
            params = float(info[1]) / 1e6 if len(info) > 1 else math.nan
            gflops = float(info[3]) if len(info) > 3 else math.nan
            return params, gflops
    except Exception:
        pass
    # Manual fallback for torch model.
    try:
        model = getattr(yolo_model, "model", yolo_model)
        params = sum(p.numel() for p in model.parameters()) / 1e6
        return float(params), math.nan
    except Exception:
        return math.nan, math.nan


def label_path_for_image(root: Path, split: str, image_path: Path) -> Path:
    return root / "labels" / split / f"{image_path.stem}.txt"


def count_gt_label(lbl: Path, target_cls: int) -> int:
    if not lbl.exists():
        return 0
    n = 0
    for ln in lbl.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = ln.split()
        if len(parts) >= 5:
            try:
                if int(float(parts[0])) == int(target_cls):
                    n += 1
            except Exception:
                continue
    return n


def summarize_count(df: pd.DataFrame) -> Dict[str, float]:
    if df.empty:
        return {
            "MAE count": math.nan,
            "RMSE count": math.nan,
            "MAPE count": math.nan,
            "Counting Accuracy": math.nan,
            "Count Bias": math.nan,
            "Over-count error": math.nan,
            "Missed detection error": math.nan,
            "Exact match rate": math.nan,
        }
    err = df["pred_count"].astype(float) - df["gt_count"].astype(float)
    abs_err = err.abs()
    mape = float((abs_err / df["gt_count"].clip(lower=1).astype(float)).mean() * 100.0)
    return {
        "MAE count": float(abs_err.mean()),
        "RMSE count": float(np.sqrt((err ** 2).mean())),
        "MAPE count": mape,
        "Counting Accuracy": max(0.0, 100.0 - mape),
        "Count Bias": float(err.mean()),
        "Over-count error": float(np.maximum(err, 0).mean()),
        "Missed detection error": float(np.maximum(-err, 0).mean()),
        "Exact match rate": float((abs_err == 0).mean()),
    }


# ============================================================
# 6. Train / validate / count-evaluate one model
# ============================================================

def find_run_best_last(model_key: str) -> Tuple[Optional[Path], Optional[Path]]:
    run_name = f"{model_key}_{DATASET_NAME}"
    wdir = RUN_DIR / run_name / "weights"
    best = wdir / "best.pt"
    last = wdir / "last.pt"
    return (best if best.exists() else None, last if last.exists() else None)


def export_best(model_key: str, best_path: Path, group: str) -> Path:
    dst_dir = SOTA_ARTIFACT_DIR if group == "sota" else ARTIFACT_DIR
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f"{model_key}_{DATASET_NAME}_best.pt"
    if best_path.resolve() != dst.resolve():
        shutil.copy2(best_path, dst)
    return dst


def load_aug_hyp() -> Dict[str, Any]:
    aug_keys = {"hsv_h", "hsv_s", "hsv_v", "degrees", "translate", "scale", "shear", "perspective", "flipud", "fliplr", "mosaic", "close_mosaic", "mixup", "copy_paste", "erasing", "auto_augment"}
    for name in [f"{DATASET_NAME}_aug.yaml", "finetune_data_1_aug.yaml", "finetune_data_aug.yaml"]:
        p = PROJECT_ROOT / "configs" / "aug" / name
        if p.exists():
            try:
                raw = yaml.safe_load(p.read_text(encoding="utf-8")) or {}
                out = {k: v for k, v in raw.items() if k in aug_keys}
                LOGGER.info("Loaded augmentation hyp from %s: %s", p, out)
                return out
            except Exception as e:
                LOGGER.warning("Could not read augmentation file %s: %s", p, repr(e))
    LOGGER.info("No custom augmentation YAML found. Using Ultralytics defaults.")
    return {}


def train_or_reuse_model(spec: Dict[str, Any], custom_modules_ok: bool, aug_hyp: Dict[str, Any]) -> Dict[str, Any]:
    from ultralytics import YOLO

    model_key = spec["model_key"]
    model_name = spec["model_name"]
    row: Dict[str, Any] = {
        "Model": model_name,
        "model_key": model_key,
        "group": spec.get("group", ""),
        "status": "unknown",
        "source": "",
        "source_kind": "",
        "run_dir": str(RUN_DIR / f"{model_key}_{DATASET_NAME}"),
        "best_pt": "",
        "last_pt": "",
        "artifact_best": "",
        "eval_split": "",
        "Params M": math.nan,
        "GFLOPs": math.nan,
        "Precision": math.nan,
        "Recall": math.nan,
        "mAP@0.5": math.nan,
        "mAP@0.5:0.95": math.nan,
        "train_seconds": math.nan,
        "val_seconds": math.nan,
        "error": "",
    }

    if model_key not in MODEL_KEYS:
        row["status"] = "skipped_not_selected"
        return row
    if spec.get("external_path_required") and not spec.get("aliases"):
        row["status"] = "skipped_missing_external_path"
        row["error"] = f"Set {model_key.upper()}_PATH env var if you have the exact implementation/weights."
        return row
    if spec.get("group") == "external_sota" and not INCLUDE_EXTERNAL_SOTA:
        row["status"] = "skipped_external_disabled"
        return row

    initial_source, source_kind = resolve_initial_source(spec)
    row["source"] = str(initial_source or "")
    row["source_kind"] = source_kind
    if not initial_source:
        row["status"] = f"skipped_{source_kind}"
        row["error"] = f"No source/checkpoint found for {model_key}."
        return row

    if yaml_needs_custom_modules(initial_source) and not custom_modules_ok:
        row["status"] = "skipped_custom_modules_not_registered"
        row["error"] = "FruitNet custom modules could not be registered."
        return row

    best, last = find_run_best_last(model_key)
    # If a final artifact already exists and FORCE_RETRAIN=0, reuse it.
    existing_artifact = first_existing([ARTIFACT_DIR / f"{model_key}_{DATASET_NAME}_best.pt", SOTA_ARTIFACT_DIR / f"{model_key}_{DATASET_NAME}_best.pt"])
    final_model_path: Optional[Path] = existing_artifact

    try:
        train_start = time.time()
        if RUN_TRAINING and has_train_and_val() and (FORCE_RETRAIN or final_model_path is None):
            LOGGER.info("Training %s (%s) from source=%s", model_name, model_key, initial_source)
            run_name = f"{model_key}_{DATASET_NAME}"
            if RESUME_TRAINING and last and not FORCE_RETRAIN:
                LOGGER.info("Resuming %s from %s", model_key, last)
                YOLO(str(last)).train(resume=True)
            else:
                model = YOLO(str(initial_source))
                model.train(
                    data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH, epochs=EPOCHS,
                    workers=WORKERS, device=DEVICE, seed=SEED, patience=PATIENCE,
                    project=str(RUN_DIR), name=run_name, exist_ok=True, pretrained=True,
                    plots=True, save=True, save_period=env_int("SAVE_PERIOD", 10),
                    **aug_hyp,
                )
            best, last = find_run_best_last(model_key)
            if best:
                final_model_path = export_best(model_key, best, spec.get("group", ""))
            elif last:
                final_model_path = last
        elif final_model_path is not None:
            LOGGER.info("Reusing existing checkpoint for %s: %s", model_key, final_model_path)
        elif best:
            final_model_path = export_best(model_key, best, spec.get("group", ""))
        else:
            # Last chance: use initial source for validation if it is a local or alias checkpoint.
            final_model_path = Path(str(initial_source)) if Path(str(initial_source)).exists() else None
            if final_model_path is None and str(initial_source).endswith(".pt"):
                # Ultralytics alias string such as yolo11n.pt; use as model identifier.
                final_model_path = Path(str(initial_source))
        row["train_seconds"] = float(time.time() - train_start)

        best2, last2 = find_run_best_last(model_key)
        row["best_pt"] = str(best2 or "")
        row["last_pt"] = str(last2 or "")
        row["artifact_best"] = str(final_model_path or "")

        if final_model_path is None:
            row["status"] = "skipped_no_final_model"
            row["error"] = "Training did not produce best/last and no existing artifact is available."
            return row

        # Complexity and validation.
        yolo = YOLO(str(final_model_path))
        row["Params M"], row["GFLOPs"] = model_complexity_from_yolo(yolo)
        if RUN_VALIDATION:
            eval_split = resolve_eval_split(EVAL_SPLIT_REQUESTED)
            if eval_split is None:
                row["status"] = "skipped_no_eval_split"
                return row
            row["eval_split"] = eval_split
            val_start = time.time()
            metrics = yolo.val(
                data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH,
                workers=WORKERS, device=DEVICE, split=eval_split,
                project=str(VAL_DIR), name=f"{model_key}_{DATASET_NAME}_{eval_split}",
                exist_ok=True, plots=True,
            )
            row.update(metrics_to_dict(metrics))
            row["val_seconds"] = float(time.time() - val_start)
        row["status"] = "ok"
        return row
    except Exception as e:
        row["status"] = "failed"
        row["error"] = repr(e)
        LOGGER.exception("Model failed: %s", model_key)
        return row


def count_eval_model(row: Dict[str, Any]) -> Tuple[Dict[str, Any], pd.DataFrame]:
    from ultralytics import YOLO

    out: Dict[str, Any] = {"model_key": row.get("model_key"), "Model": row.get("Model"), "count_status": "unknown"}
    if row.get("status") != "ok":
        out["count_status"] = "skipped_model_not_ok"
        return out, pd.DataFrame()
    model_path = row.get("artifact_best") or row.get("best_pt") or row.get("source")
    if not model_path:
        out["count_status"] = "skipped_no_model_path"
        return out, pd.DataFrame()
    split = resolve_eval_split(EVAL_SPLIT_REQUESTED)
    if split is None:
        out["count_status"] = "skipped_no_split"
        return out, pd.DataFrame()
    root = Path(DATA_CFG["path"])
    split_rel = DATA_CFG.get(split, f"images/{split}")
    images = list(iter_images(root / str(split_rel)))
    if MAX_COUNT_IMAGES > 0:
        images = images[:MAX_COUNT_IMAGES]
    if not images:
        out["count_status"] = "skipped_no_images"
        return out, pd.DataFrame()

    LOGGER.info("Counting eval for %s on %d images split=%s", row.get("model_key"), len(images), split)
    model = YOLO(str(model_path))
    records: List[Dict[str, Any]] = []
    t0 = time.time()
    for idx, img in enumerate(images, start=1):
        try:
            pred_start = time.time()
            r = model.predict(str(img), imgsz=IMG_SIZE, conf=COUNT_CONF, iou=COUNT_IOU, verbose=False)[0]
            pred_ms = (time.time() - pred_start) * 1000.0
            pred_count = 0
            n_total = 0
            if r.boxes is not None and r.boxes.cls is not None:
                cls = r.boxes.cls.detach().cpu().numpy().astype(int).tolist()
                n_total = len(cls)
                pred_count = int(sum(1 for c in cls if int(c) == int(TARGET_CLS)))
            gt = count_gt_label(label_path_for_image(root, split, img), TARGET_CLS)
            records.append({
                "image": str(img),
                "image_name": img.name,
                "split": split,
                "gt_count": int(gt),
                "pred_count": int(pred_count),
                "pred_total_boxes": int(n_total),
                "abs_err": abs(int(pred_count) - int(gt)),
                "sq_err": (int(pred_count) - int(gt)) ** 2,
                "ape": abs(int(pred_count) - int(gt)) / max(int(gt), 1),
                "predict_ms": float(pred_ms),
                "conf": COUNT_CONF,
                "iou": COUNT_IOU,
                "target_cls": TARGET_CLS,
            })
            if idx % 25 == 0 or idx == len(images):
                partial_df = pd.DataFrame(records)
                atomic_write_csv(COUNT_DIR / f"{row.get('model_key')}_count_eval_partial.csv", partial_df)
        except Exception as e:
            records.append({
                "image": str(img), "image_name": img.name, "split": split,
                "gt_count": count_gt_label(label_path_for_image(root, split, img), TARGET_CLS),
                "pred_count": math.nan, "error": repr(e), "conf": COUNT_CONF, "iou": COUNT_IOU,
                "target_cls": TARGET_CLS,
            })
            LOGGER.warning("Count eval failed for %s image=%s: %s", row.get("model_key"), img, repr(e))
    df = pd.DataFrame(records)
    df_ok = df.dropna(subset=["pred_count"]).copy()
    metrics = summarize_count(df_ok)
    out.update(metrics)
    out["count_status"] = "ok"
    out["count_split"] = split
    out["n_count_images"] = int(len(df_ok))
    out["avg_predict_ms"] = float(df_ok["predict_ms"].mean()) if "predict_ms" in df_ok and len(df_ok) else math.nan
    out["count_eval_seconds"] = float(time.time() - t0)
    atomic_write_csv(COUNT_DIR / f"{row.get('model_key')}_count_eval.csv", df)
    return out, df


# ============================================================
# 7. RL / yield merge for FruitNet GCSR
# ============================================================

def find_latest(pattern: str) -> Optional[Path]:
    hits = sorted((OUTPUT_ROOT / "results").glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[0] if hits else None


def load_rl_row() -> Dict[str, Any]:
    p = Path(os.environ.get("RL_TABLE", "")).resolve() if os.environ.get("RL_TABLE", "").strip() else None
    if p is None or not p.exists():
        p = find_latest(f"exp03_rl*{DATASET_NAME}*/table_4_4_1_postprocess_comparison.csv")
    if not p or not p.exists():
        LOGGER.warning("No RL table found for GCSR row.")
        return {}
    df = safe_read_csv(p)
    if df.empty:
        return {}
    method_col = "Method" if "Method" in df.columns else ("method" if "method" in df.columns else None)
    if method_col:
        cand = df[df[method_col].astype(str).str.contains("rl", case=False, na=False)]
        if cand.empty:
            cand = df.tail(1)
    else:
        cand = df.tail(1)
    row = cand.iloc[0].to_dict()
    row["_source_rl_table"] = str(p)
    return row


def load_yield_row() -> Dict[str, Any]:
    p = Path(os.environ.get("YIELD_TABLE", "")).resolve() if os.environ.get("YIELD_TABLE", "").strip() else None
    if p is None or not p.exists():
        p = find_latest(f"exp04_yield*{DATASET_NAME}*/table_yield_estimation.csv")
    if not p or not p.exists():
        LOGGER.warning("No yield table found for GCSR row.")
        return {}
    df = safe_read_csv(p)
    if df.empty:
        return {}
    method_col = "method" if "method" in df.columns else ("Method" if "Method" in df.columns else None)
    if method_col:
        cand = df[df[method_col].astype(str).str.contains("gcsr|rl", case=False, na=False, regex=True)]
        if cand.empty:
            cand = df.tail(1)
    else:
        cand = df.tail(1)
    row = cand.iloc[0].to_dict()
    row["_source_yield_table"] = str(p)
    return row


def get_case_insensitive(row: Dict[str, Any], *names: str) -> Any:
    lower = {str(k).lower(): k for k in row.keys()}
    for n in names:
        k = lower.get(n.lower())
        if k is not None:
            return row.get(k)
    return None


def add_gcsr_row(comp: pd.DataFrame) -> pd.DataFrame:
    rl = load_rl_row()
    if not rl:
        return comp
    base = comp[comp["model_key"].astype(str).map(normalize_model_key) == "fruitnet_gcs"].tail(1)
    if not base.empty:
        gcsr = base.iloc[0].to_dict()
    else:
        gcsr = {"Model": "FruitNet GCSR", "model_key": "fruitnet_gcsr", "group": "fruitnet_rl"}
    gcsr["Model"] = "FruitNet GCSR"
    gcsr["model_key"] = "fruitnet_gcsr"
    gcsr["group"] = "fruitnet_rl"
    gcsr["status"] = "ok_from_rl"
    # RL metrics may be uppercase from 07 or lowercase from older scripts.
    mapping = {
        "MAE count": ["MAE count", "MAE_count", "mae", "mae_count"],
        "RMSE count": ["RMSE count", "RMSE_count", "rmse", "rmse_count"],
        "MAPE count": ["MAPE count", "MAPE_count", "mape", "mape_count"],
        "Counting Accuracy": ["Counting Accuracy", "Counting_Accuracy", "counting_accuracy"],
        "Duplicate error": ["duplicate_error", "Duplicate error", "duplicate_iou_ratio"],
        "Missed detection error": ["missed_detection_error", "missed_count_error", "Missed detection error"],
        "Postprocess latency ms": ["postprocess_latency_ms", "Postprocess latency ms"],
    }
    for dst, names in mapping.items():
        v = get_case_insensitive(rl, *names)
        if v is not None:
            gcsr[dst] = v
    y = load_yield_row()
    ymap = {
        "MAPE yield": ["MAPE yield", "mape_yield", "mape_yield_%", "MAPE_yield"],
        "MAE yield g": ["MAE yield g", "mae_yield_g", "MAE_yield_g"],
        "RMSE yield g": ["RMSE yield g", "rmse_yield_g", "RMSE_yield_g"],
        "Yield R2": ["Yield R2", "r2", "R2", "yield_r2"],
    }
    for dst, names in ymap.items():
        v = get_case_insensitive(y, *names)
        if v is not None:
            gcsr[dst] = v
    return pd.concat([comp, pd.DataFrame([gcsr])], ignore_index=True)


# ============================================================
# 8. Main
# ============================================================

def main() -> None:
    t_all = time.time()
    write_status("running", {"log_path": str(LOG_PATH)})
    LOGGER.info("PROJECT_ROOT=%s", PROJECT_ROOT)
    LOGGER.info("DATA_DIR=%s", DATA_DIR)
    LOGGER.info("OUTPUT_ROOT=%s", OUTPUT_ROOT)
    LOGGER.info("DATA_YAML=%s", DATA_YAML)
    LOGGER.info("MODEL_KEYS=%s", MODEL_KEYS)
    LOGGER.info("RUN_TRAINING=%s RUN_VALIDATION=%s RUN_COUNT_EVAL=%s FORCE_RETRAIN=%s", RUN_TRAINING, RUN_VALIDATION, RUN_COUNT_EVAL, FORCE_RETRAIN)

    custom_ok = register_custom_modules()
    aug_hyp = load_aug_hyp()
    registry = [m for m in model_registry() if m["model_key"] in MODEL_KEYS]
    atomic_write_json(REGISTRY_DIR / "model_registry_requested.json", registry)

    if RUN_TRAINING and not has_train_and_val():
        LOGGER.warning("Training requested, but train/val split is missing or empty. Models will be reused/skipped.")

    progress = load_progress()
    model_rows: List[Dict[str, Any]] = []
    count_rows: List[Dict[str, Any]] = []
    audit_rows: List[Dict[str, Any]] = []

    for i, spec in enumerate(registry, start=1):
        key = spec["model_key"]
        LOGGER.info("========== [%d/%d] %s ==========" , i, len(registry), key)
        write_model_status(key, {"status": "running", "spec": spec})
        row = train_or_reuse_model(spec, custom_ok, aug_hyp)
        model_rows.append(row)
        audit_rows.append({**spec, **{k: row.get(k) for k in ["status", "source", "source_kind", "artifact_best", "best_pt", "error"]}})
        write_model_status(key, row)
        atomic_write_csv(EXP_DIR / "sota_train_eval_status.csv", pd.DataFrame(model_rows))
        atomic_write_csv(EXP_DIR / "sota_model_audit.csv", pd.DataFrame(audit_rows))
        progress[key] = {"train_eval_status": row.get("status"), "artifact_best": row.get("artifact_best"), "updated_at": now()}
        save_progress(progress)

        if RUN_COUNT_EVAL and row.get("status") == "ok":
            cmetrics, _ = count_eval_model(row)
            count_rows.append(cmetrics)
            atomic_write_csv(EXP_DIR / "sota_counting_metrics.csv", pd.DataFrame(count_rows))
            progress[key]["count_status"] = cmetrics.get("count_status")
            save_progress(progress)

    det_df = pd.DataFrame(model_rows)
    count_df = pd.DataFrame(count_rows)
    if not count_df.empty and not det_df.empty:
        count_df["model_key"] = count_df["model_key"].map(normalize_model_key)
        det_df["model_key"] = det_df["model_key"].map(normalize_model_key)
        merge_cols = [c for c in count_df.columns if c not in {"Model"}]
        comp = det_df.merge(count_df[merge_cols], on="model_key", how="left")
    else:
        comp = det_df.copy()

    comp = add_gcsr_row(comp)

    preferred = [
        "Model", "model_key", "group", "status", "Params M", "GFLOPs",
        "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95",
        "MAE count", "RMSE count", "MAPE count", "Counting Accuracy",
        "Duplicate error", "Missed detection error", "Postprocess latency ms",
        "MAPE yield", "MAE yield g", "RMSE yield g", "Yield R2",
        "artifact_best", "best_pt", "eval_split", "count_split", "avg_predict_ms",
        "train_seconds", "val_seconds", "count_eval_seconds", "error",
    ]
    comp = comp[[c for c in preferred if c in comp.columns] + [c for c in comp.columns if c not in preferred]]

    # Main paper table: keep real completed rows plus GCSR row, not skipped external placeholders.
    paper = comp[comp["status"].astype(str).isin(["ok", "ok_from_rl"])].copy() if "status" in comp.columns else comp.copy()
    if STRICT_EXTERNAL_SOTA:
        required = {"pepper_yolo", "gae_yolo"}
        missing = required.difference(set(paper["model_key"].astype(str)))
        if missing:
            raise RuntimeError(f"STRICT_EXTERNAL_SOTA=1 but missing exact external SOTA models: {sorted(missing)}")

    atomic_write_csv(EXP_DIR / "table_4_5_sota_comparison_full.csv", comp)
    atomic_write_csv(EXP_DIR / "table_4_5_sota_comparison.csv", paper)
    atomic_write_text(EXP_DIR / "table_4_5_sota_comparison.md", paper.to_markdown(index=False))
    atomic_write_json(EXP_DIR / "input_manifest.json", {
        "created_at": now(),
        "project_root": str(PROJECT_ROOT),
        "data_dir": str(DATA_DIR),
        "data_yaml": str(DATA_YAML),
        "output_root": str(OUTPUT_ROOT),
        "exp_dir": str(EXP_DIR),
        "model_keys": MODEL_KEYS,
        "run_training": RUN_TRAINING,
        "run_validation": RUN_VALIDATION,
        "run_count_eval": RUN_COUNT_EVAL,
        "force_retrain": FORCE_RETRAIN,
        "resume_training": RESUME_TRAINING,
        "eval_split_requested": EVAL_SPLIT_REQUESTED,
        "target_cls": TARGET_CLS,
        "class_names": CLASS_NAMES_LIST,
        "img_size": IMG_SIZE,
        "batch": BATCH,
        "epochs": EPOCHS,
        "device": DEVICE,
        "count_conf": COUNT_CONF,
        "count_iou": COUNT_IOU,
        "max_count_images": MAX_COUNT_IMAGES,
        "log_path": str(LOG_PATH),
    })
    write_status("complete", {
        "table": str(EXP_DIR / "table_4_5_sota_comparison.csv"),
        "full_table": str(EXP_DIR / "table_4_5_sota_comparison_full.csv"),
        "audit": str(EXP_DIR / "sota_model_audit.csv"),
        "elapsed_seconds": float(time.time() - t_all),
    })
    LOGGER.info("[09 full] Done. EXP_DIR=%s elapsed=%.1fs", EXP_DIR, time.time() - t_all)
    print("EXP_DIR:", EXP_DIR)
    print("Main table:", EXP_DIR / "table_4_5_sota_comparison.csv")
    print("Full/audit table:", EXP_DIR / "table_4_5_sota_comparison_full.csv")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        LOGGER.exception("Fatal error in 09_sota_comparison_full")
        write_status("failed", {"error": repr(e), "traceback": traceback.format_exc(), "log_path": str(LOG_PATH)})
        raise


2026-07-03 15:14:34,623 | INFO | PROJECT_ROOT=/home/diy-hus/fruitnet-chili-yield
2026-07-03 15:14:34,624 | INFO | DATA_DIR=/mnt/f/fruitnet-chili-yield-data/finetune_data
2026-07-03 15:14:34,625 | INFO | OUTPUT_ROOT=/mnt/f/fruitnet-chili-yield-outputs
2026-07-03 15:14:34,626 | INFO | DATA_YAML=/home/diy-hus/fruitnet-chili-yield/configs/data/finetune_data.yaml
2026-07-03 15:14:34,627 | INFO | MODEL_KEYS=['yolov5s', 'yolov8n', 'yolov10n', 'yolo11n', 'fruitnet_b', 'fruitnet_g', 'fruitnet_gc', 'fruitnet_gcs', 'pepper_yolo', 'gae_yolo']
2026-07-03 15:14:34,628 | INFO | RUN_TRAINING=True RUN_VALIDATION=True RUN_COUNT_EVAL=True FORCE_RETRAIN=False
2026-07-03 15:14:37,097 | INFO | Registered FruitNet custom modules.
2026-07-03 15:14:37,098 | INFO | No custom augmentation YAML found. Using Ultralytics defaults.
2026-07-03 15:14:56,345 | INFO | ========== [1/10] yolov5s ==========
2026-07-03 15:15:16,913 | INFO | Reusing existing checkpoint for yolov5s: /mnt/f/fruitnet-chili-yield-outputs/artifac